# Proyecto Sprint 10: Recomendación de planes Megaline

El objetivo del proyecto es desarrollar un modelo de clasificación que, a partir del comportamiento mensual de los clientes, recomiende uno de los nuevos planes de Megaline: Smart o Ultra.

La variable objetivo es `is_ultra`, donde:
- `0` representa el plan Smart.
- `1` representa el plan Ultra.

El objetivo es obtener una exactitud de al menos 0.75 en el conjunto de prueba.

## 1. Exploración inicial de los datos

Se carga el dataset y se revisan su estructura, tipos de datos, valores ausentes y posibles duplicados antes de construir los modelos.

In [14]:
import pandas as pd

df = pd.read_csv('users_behavior.csv')

display(df.head())

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [16]:
print(df.isna().sum())
print('Duplicados:', df.duplicated().sum())

calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64
Duplicados: 0


## 2. Separación de características y objetivo

La variable objetivo es `is_ultra`, ya que indica el plan actual del cliente:

- `0` = Smart
- `1` = Ultra

Las demás columnas (`calls`, `minutes`, `messages` y `mb_used`) se utilizarán como características para predecir el plan.

In [17]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

print(features.shape)
print(target.shape)

(3214, 4)
(3214,)


## 3. División de los datos

Los datos se dividirán en tres conjuntos:

- 60% para entrenamiento.
- 20% para validación.
- 20% para prueba.

El conjunto de entrenamiento se utilizará para entrenar los modelos, el de validación para comparar modelos e hiperparámetros, y el conjunto de prueba se reservará para la evaluación final.

In [18]:
from sklearn.model_selection import train_test_split

features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.40,
    random_state=12345
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.50,
    random_state=12345
)

print('Entrenamiento:')
print(features_train.shape)
print(target_train.shape)

print('\nValidación:')
print(features_valid.shape)
print(target_valid.shape)

print('\nPrueba:')
print(features_test.shape)
print(target_test.shape)

Entrenamiento:
(1928, 4)
(1928,)

Validación:
(643, 4)
(643,)

Prueba:
(643, 4)
(643,)


## 4. Evaluación de modelos

Primero se evaluará un árbol de decisión con diferentes valores de `max_depth`.

El modelo se entrenará con el conjunto de entrenamiento y se evaluará con el conjunto de validación. El objetivo es encontrar una profundidad que logre buena exactitud sin sobreajustar el modelo.

### 4.1 Árbol de decisión

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_tree_model = None
best_tree_score = 0
best_tree_depth = 0

for depth in range(1, 11):
    model = DecisionTreeClassifier(
        random_state=12345,
        max_depth=depth
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    score = accuracy_score(target_valid, predictions_valid)

    print('max_depth =', depth, '| accuracy =', score)

    if score > best_tree_score:
        best_tree_model = model
        best_tree_score = score
        best_tree_depth = depth

max_depth = 1 | accuracy = 0.7542768273716952
max_depth = 2 | accuracy = 0.7822706065318819
max_depth = 3 | accuracy = 0.7853810264385692
max_depth = 4 | accuracy = 0.7791601866251944
max_depth = 5 | accuracy = 0.7791601866251944
max_depth = 6 | accuracy = 0.7838258164852255
max_depth = 7 | accuracy = 0.7822706065318819
max_depth = 8 | accuracy = 0.7791601866251944
max_depth = 9 | accuracy = 0.7822706065318819
max_depth = 10 | accuracy = 0.7744945567651633


### Resultado del árbol de decisión

Se probaron valores de `max_depth` entre 1 y 10.

La mejor exactitud en el conjunto de validación fue aproximadamente **0.7854**, obtenida con `max_depth=3`.

Aumentar la profundidad por encima de este valor no produjo una mejora consistente en la exactitud de validación, por lo que se conserva la profundidad 3 como la mejor configuración encontrada para este modelo.

In [20]:
print('Mejor profundidad:', best_tree_depth)
print('Mejor accuracy:', best_tree_score)

Mejor profundidad: 3
Mejor accuracy: 0.7853810264385692


### 4.2 Bosque aleatorio

A continuación se evaluará un modelo Random Forest. Se probarán diferentes cantidades de árboles mediante el hiperparámetro `n_estimators`.

Cada modelo se entrenará con el conjunto de entrenamiento y se comparará mediante su exactitud en el conjunto de validación.

In [21]:
from sklearn.ensemble import RandomForestClassifier

best_forest_model = None
best_forest_score = 0
best_forest_est = 0

for est in range(10, 101, 10):
    model = RandomForestClassifier(
        random_state=12345,
        n_estimators=est
    )

    model.fit(features_train, target_train)

    predictions_valid = model.predict(features_valid)

    score = accuracy_score(target_valid, predictions_valid)

    print('n_estimators =', est, '| accuracy =', score)

    if score > best_forest_score:
        best_forest_model = model
        best_forest_score = score
        best_forest_est = est

n_estimators = 10 | accuracy = 0.7853810264385692
n_estimators = 20 | accuracy = 0.7869362363919129
n_estimators = 30 | accuracy = 0.7838258164852255
n_estimators = 40 | accuracy = 0.7838258164852255
n_estimators = 50 | accuracy = 0.7916018662519441
n_estimators = 60 | accuracy = 0.7853810264385692
n_estimators = 70 | accuracy = 0.7791601866251944
n_estimators = 80 | accuracy = 0.7822706065318819
n_estimators = 90 | accuracy = 0.7838258164852255
n_estimators = 100 | accuracy = 0.7853810264385692


In [22]:
print('Mejor número de árboles:', best_forest_est)
print('Mejor accuracy:', best_forest_score)

Mejor número de árboles: 50
Mejor accuracy: 0.7916018662519441


### Resultado del bosque aleatorio

Se probaron modelos Random Forest con entre 10 y 100 árboles, aumentando `n_estimators` en intervalos de 10.

La mejor exactitud obtenida en el conjunto de validación fue aproximadamente **0.7916**, utilizando `n_estimators=50`.

Este resultado supera al mejor árbol de decisión evaluado hasta ahora, cuya exactitud fue aproximadamente **0.7854**.

### 4.3 Regresión logística

Finalmente se evaluará un modelo de regresión logística. Este modelo servirá como una alternativa más simple a los modelos basados en árboles.

El modelo se entrenará con el conjunto de entrenamiento y su exactitud se medirá en el conjunto de validación.

In [23]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    random_state=12345,
    solver='liblinear'
)

logistic_model.fit(features_train, target_train)

logistic_predictions = logistic_model.predict(features_valid)

logistic_score = accuracy_score(
    target_valid,
    logistic_predictions
)

print('Accuracy de regresión logística:', logistic_score)

Accuracy de regresión logística: 0.7091757387247278


### Comparación de modelos

Se compararon tres modelos de clasificación:

- Árbol de decisión: exactitud aproximada de **0.7854**.
- Bosque aleatorio: exactitud aproximada de **0.7916**.
- Regresión logística: exactitud aproximada de **0.7092**.

El modelo con mejor desempeño en el conjunto de validación fue el **bosque aleatorio**, utilizando `n_estimators=50`.

Por lo tanto, este modelo será seleccionado para la evaluación final con el conjunto de prueba.

## 5. Evaluación final en el conjunto de prueba

Después de seleccionar el modelo con mejor desempeño en validación, se evalúa una sola vez utilizando el conjunto de prueba.

Este conjunto no se utilizó durante el entrenamiento ni durante la selección de hiperparámetros, por lo que permite estimar el rendimiento del modelo sobre datos no vistos.

In [24]:
test_predictions = best_forest_model.predict(features_test)

test_accuracy = accuracy_score(
    target_test,
    test_predictions
)

print('Accuracy en el conjunto de prueba:', test_accuracy)

Accuracy en el conjunto de prueba: 0.7931570762052877


### Resultado en el conjunto de prueba

El modelo Random Forest obtuvo una exactitud aproximada de **0.7932** en el conjunto de prueba.

Esto significa que el modelo clasificó correctamente cerca del **79.3%** de los usuarios.

El resultado supera el umbral mínimo de exactitud de **0.75** establecido para el proyecto.

In [25]:
print(target_train.value_counts())
print(target_train.value_counts(normalize=True))

is_ultra
0    1335
1     593
Name: count, dtype: int64
is_ultra
0    0.692427
1    0.307573
Name: proportion, dtype: float64


Antes de crear el modelo de referencia, se revisa la distribución de la variable objetivo para identificar la clase mayoritaria.

## 6. Prueba de cordura

Para comprobar que el modelo realmente aprende patrones útiles, se compara su exactitud con la de un clasificador simple que siempre predice la clase más frecuente.

Si el modelo entrenado supera claramente a este clasificador de referencia, se considera que pasa la prueba de cordura.

In [26]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy='most_frequent')

dummy_model.fit(features_train, target_train)

dummy_predictions = dummy_model.predict(features_test)

dummy_accuracy = accuracy_score(
    target_test,
    dummy_predictions
)

print('Accuracy del modelo de referencia:', dummy_accuracy)
print('Accuracy del Random Forest:', test_accuracy)

Accuracy del modelo de referencia: 0.6842923794712286
Accuracy del Random Forest: 0.7931570762052877


### Resultado de la prueba de cordura

El clasificador de referencia obtuvo una exactitud aproximada de **0.6843**, mientras que el modelo Random Forest alcanzó una exactitud de **0.7932** en el conjunto de prueba.

Como el modelo entrenado supera claramente al clasificador que siempre predice la clase más frecuente, se considera que el modelo pasa la prueba de cordura.

## 7. Conclusión general

El objetivo del proyecto fue desarrollar un modelo de clasificación capaz de recomendar uno de los nuevos planes de Megaline, Smart o Ultra, a partir del comportamiento mensual de los clientes.

El dataset contenía 3214 observaciones y no presentaba valores ausentes ni duplicados. Las características utilizadas fueron `calls`, `minutes`, `messages` y `mb_used`, mientras que `is_ultra` se utilizó como variable objetivo.

Los datos se dividieron en tres conjuntos:

- 60% para entrenamiento.
- 20% para validación.
- 20% para prueba.

Se evaluaron tres algoritmos de clasificación:

- Árbol de decisión: mejor exactitud de validación aproximada de **0.7854**, con `max_depth=3`.
- Bosque aleatorio: mejor exactitud de validación aproximada de **0.7916**, con `n_estimators=50`.
- Regresión logística: exactitud de validación aproximada de **0.7092**.

El modelo con mejor desempeño fue el **Random Forest**, por lo que se utilizó para la evaluación final.

En el conjunto de prueba, el modelo obtuvo una exactitud aproximada de **0.7932**, superando el umbral mínimo de **0.75** establecido para el proyecto.

Finalmente, se realizó una prueba de cordura utilizando un `DummyClassifier` que siempre predice la clase más frecuente. Este modelo de referencia obtuvo una exactitud aproximada de **0.6843**, claramente inferior al resultado del Random Forest.

Por lo tanto, el modelo seleccionado supera tanto el requisito mínimo del proyecto como el modelo de referencia, lo que indica que aprendió patrones útiles para recomendar entre los planes Smart y Ultra.